# GPIO

In [1]:
 pip install pyserial

Note: you may need to restart the kernel to use updated packages.


In [1]:
import serial 
import time 

ser = serial.Serial('/dev/ttyUSB0',baudrate=115200,bytesize =8, parity ='N', stopbits =1)
hex = '3A0100020003000400'
#hex = '3A0101020103010401'

data_send = bytes.fromhex(hex)
ser.write(data_send)

ser.close()


# USe this for demo

In [2]:
import numpy as np
import cv2
import serial
import time

# Initialize PySerial
ser = serial.Serial('/dev/ttyUSB0', 115200)  # Replace with your serial port and baud rate

webcam = cv2.VideoCapture(2)

# Function to send command via PySerial
def send_command(command):
    ser.write(bytes.fromhex(command))
    time.sleep(0.2)  # Adjust delay as necessary

# Define commands
initial_command = '3A0100020003000400'

# Initialize variables to track state
current_location = 'red'  # Start at red
previous_location = 'red'
last_sent_time = time.time()

def calibrate_hsv_ranges(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    color_ranges = {
        'red': [(136, 87, 111), (180, 255, 255)],
        'green': [(35, 100, 50), (85, 255, 255)],
        'blue': [(100, 100, 100), (140, 255, 255)]
    }
    return hsv, color_ranges

# Start a while loop 
while True:
    _, imageFrame = webcam.read()

    # Calibrate HSV ranges
    hsvFrame, color_ranges = calibrate_hsv_ranges(imageFrame)

    # Dictionary to hold masks and results for each color
    masks = {}
    kernel = np.ones((5, 5), "uint8")

    # Variable to count detected colors
    detected_colors = 0

    for color, (lower, upper) in color_ranges.items():
        # Create mask for the color
        mask = cv2.inRange(hsvFrame, np.array(lower, np.uint8), np.array(upper, np.uint8))
        
        # Apply morphological operations
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.dilate(mask, kernel)
        
        # Store masks
        masks[color] = mask

    # Define color bounding box color (BGR format)
    color_bbox = {
        'red': (0, 0, 255),
        'green': (0, 255, 0),
        'blue': (255, 0, 0)
    }

    detected_color = None

    for color, mask in masks.items():
        # Find contours for each color
        contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        
        for contour in contours:
            area = cv2.contourArea(contour)
            if area > 300:  # Adjust this threshold as needed
                x, y, w, h = cv2.boundingRect(contour)
                cv2.rectangle(imageFrame, (x, y), (x + w, y + h), color_bbox[color], 2)
                cv2.putText(imageFrame, f"{color.capitalize()} Colour", (x, y), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color_bbox[color])

                detected_colors += 1
                detected_color = color

    if detected_colors == 1:
        # Only send command if exactly one color is detected
        if detected_color == 'red':
            if previous_location == 'blue':
                send_command('3A0101020003000400')
            elif previous_location == 'green':
                send_command('3A0101020003000400')
            elif previous_location == 'red':
                pass
            current_location = 'red'

        elif detected_color == 'green':
            if previous_location == 'blue':
                send_command('3A0100020103000400')
            elif previous_location == 'green':
                pass
            elif previous_location == 'red':
                send_command('3A0100020103000401')
            current_location = 'green'

        elif detected_color == 'blue':
            if previous_location == 'red':
                send_command('3A0100020003010401')
            elif previous_location == 'green':
                send_command('3A0100020003010401')
            elif previous_location == 'blue':
                pass
            current_location = 'blue'

        # Update previous location8888888888
        previous_location = detected_color

    # Display the resulting frame
    cv2.imshow("Multiple Color Detection in Real-Time", imageFrame)
    
    # Handle key press 'q' to exit and reset
    if cv2.waitKey(10) & 0xFF == ord('q'):
        send_command(initial_command)  # Return to initial position
        break

# Release resources
webcam.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 